# Karma scoring

For each of the top 100 characters, send their biography + their list of affiliated targets to local Gemma and ask for a 1-10 karma score per target.

This notebook is intentionally written as a record of the three iterations we went through. The goal was not only to produce karma edge data, but also to document why the first output was not good enough, what broke, and how each later version fixed a specific issue. That history matters for the final explainer notebook because the LLM scores are a derived dataset, not raw observations.

**Canonical output: `../csvs/karma_edges_v3.csv`.** The iterations below also produce `karma_edges.csv` (v1, full first pass) and `karma_edges_v2.csv` (intermediate, post-prune); only v3 is kept long-term and read by downstream notebooks. Inputs (`characters_bio.csv`, `characters_enriched_v3.csv`, `top100_outgoing_v3.csv`) all live under `../csvs/`.

The three iterations are:

1. **Full first pass**: score the original top-100 outgoing affiliation sources. This proved that the method worked, but also exposed long-response JSON failures.
2. **Chunked rerun**: rerun missing sources in smaller target batches. This fixed parse failures, but a later sanity check revealed that the input network itself was inflated by family-tree links.
3. **v2 prune, then v3 finalisation**: rebuild the target list from the cleaner scraper to produce `karma_edges_v2.csv` (prune + score new entrants), then drop corrupted rows and fill missing source-target pairs to produce the canonical `karma_edges_v3.csv`.


## Design choices

Three methodology choices shape what follows. Each has a real trade-off, and the iterations below are easier to read with the rationale in mind.

**Local Gemma 4 via Ollama, not a paid API.** An earlier proof of concept used Claude Haiku at a 0-100 scale on the top-200 characters. We moved to local Gemma for two reasons: reproducibility (anyone with Ollama can re-run the pipeline; no API key, no third-party service in the loop) and zero per-run cost. The trade-off is a smaller, less capable model that needs more validation - which is exactly what iterations 1-3 provide.

**A 1-10 scale, not three classes.** We could have asked the model directly for friend / neutral / enemy. The 1-10 scale lets the model express uncertainty (the "hedge at 5" visible in the distribution is the model saying *no narrative evidence either way*), preserves more information per call, and collapses cleanly into three classes downstream for visualisation. The trade-off is a slightly noisier signal at the per-pair level.

**Top-100 source characters, not top-N.** The number of LLM calls is the binding cost. Each top-100 character has roughly 100 affiliated targets, so the scoring step is already ~10,000 generations. Top-200 would quadruple the work for diminishing return - the long tail is mostly minor characters with sparse bios where the model has less to work with anyway.


## Iteration 1 - first full LLM pass

The first version was deliberately simple: take the top-100 characters by outgoing affiliation count, give the model each source biography plus all of that source's targets, and ask for one JSON object of scores.

This was the right starting point because it tested the core idea with minimal moving parts. If the model could not produce plausible scores here, there would be no reason to build more cleanup around it. The main weakness was that some high-degree characters had very long target lists, so the model sometimes returned malformed JSON. Those failures did not mean the scoring idea was useless; they showed that the unit of work was too large.

In [4]:
import requests
import pandas as pd
import json
import re
import csv
from tqdm.auto import tqdm

top = pd.read_csv("../csvs/top100_outgoing.csv")
bios = pd.read_csv("../csvs/characters_bio.csv")
bio_lookup = dict(zip(bios["ID"], bios["bio"].fillna("")))
name_lookup = dict(zip(bios["ID"], bios["name"].fillna(bios["ID"])))
print(len(top), "sources")


100 sources


In [5]:
def score(source_id, target_ids):
    bio = bio_lookup.get(source_id, "")
    name = name_lookup.get(source_id, source_id)

    prompt = f"""You are analyzing the biography of {name} from A Song of Ice and Fire.
For every character ID listed, give a karma score from 1 to 10 reflecting how {name} feels about them, using the bio AND your general knowledge of the books.

1 = extremely hostile (sworn enemy, killer, betrayer)
10 = extremely friendly (closest ally, beloved family, sworn friend)

Biography:
{bio}

Score these character IDs (use them EXACTLY as JSON keys):
{', '.join(target_ids)}

Return ONLY a JSON object like {{"ID_one": 7, "ID_two": 2}}. No prose, no code fences."""

    res = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "gemma4:e4b",
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": 0.4, "num_ctx": 16384, "num_predict": 6000},
        },
        timeout=900,
    )
    res.raise_for_status()
    text = res.json()["response"].strip()
    # strip code fences if present
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text)
    m = re.search(r"\{.*\}", text, re.DOTALL)
    return json.loads(m.group(0)) if m else {}


In [6]:
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

with open("../csvs/karma_edges.csv", "w", newline="") as f:
    csv.writer(f).writerow(["source_id", "target_id", "karma_score"])

write_lock = threading.Lock()

def process(row):
    sid = row.ID
    targets = [t for t in (row.outgoing_edges or "").split(";") if t]
    if not targets:
        return sid, 0
    try:
        scores = score(sid, targets)
    except Exception as e:
        print(f"{sid}: {e}")
        return sid, 0
    rows = []
    for tid, s in scores.items():
        try:
            s = max(1, min(10, int(s)))
        except (TypeError, ValueError):
            continue
        if tid in targets:
            rows.append([sid, tid, s])
    with write_lock, open("../csvs/karma_edges.csv", "a", newline="") as f:
        csv.writer(f).writerows(rows)
    return sid, len(rows)

jobs = list(top.itertuples(index=False))
with ThreadPoolExecutor(max_workers=4) as ex:
    futures = [ex.submit(process, row) for row in jobs]
    for f in tqdm(as_completed(futures), total=len(futures)):
        f.result()


  3%|▎         | 3/100 [05:58<3:10:35, 117.89s/it]

Tyrion_Lannister: Expecting ',' delimiter: line 1 column 3649 (char 3648)


  4%|▍         | 4/100 [07:17<2:43:43, 102.33s/it]

Jaime_Lannister: Expecting ',' delimiter: line 1 column 2245 (char 2244)


 16%|█▌        | 16/100 [33:43<2:43:05, 116.50s/it]

...

 47%|████▋     | 47/100 [1:24:32<1:05:52, 74.57s/it]

Daemon_Velaryon_(son_of_Aethan): Expecting ',' delimiter: line 1 column 2282 (char 2281)


 49%|████▉     | 49/100 [1:26:05<51:09, 60.19s/it]  

Otto_Hightower: Expecting ',' delimiter: line 1 column 2248 (char 2247)


100%|██████████| 100/100 [2:50:03<00:00, 102.04s/it] 


### Reflection on iteration 1

The quick look showed that the first pass produced a usable-looking score distribution and recognizable extreme examples, but it also showed incomplete coverage: only **87 of the 100** source characters received scores. The missing cases were not random small characters; they were often high-degree sources with large prompts and long JSON outputs.

The issue was therefore operational rather than conceptual. We were asking the model to do too much in one response. The next iteration keeps the same scoring definition but changes the batching strategy so the model only has to return shorter JSON objects.

In [7]:
edges = pd.read_csv("../csvs/karma_edges.csv")
print(len(edges), "edges,", edges["source_id"].nunique(), "sources")
print(edges["karma_score"].value_counts().sort_index())


9724 edges, 87 sources
karma_score
1      513
2      707
3     1543
4      807
5     3738
6     1035
7      753
8      341
9      190
10      97
Name: count, dtype: int64


## Re-run missing sources (chunked)

The first pass dropped 13 high-out-degree sources because the model returned malformed JSON when the target list got long. Here we identify which `top100` sources are missing from `karma_edges.csv` and re-score them in **chunks of 40 targets** so each JSON response stays short and parses cleanly. Results are **appended** to `karma_edges.csv`; chunk-level errors go to `karma_failures.csv`.

In [ ]:
import os
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

CHUNK = 40  # targets per LLM call - keeps JSON response well under the parse-failure threshold

# Which top100 sources are currently missing (0 edges) in karma_edges.csv?
existing = pd.read_csv("../csvs/karma_edges.csv")
have = set(existing["source_id"].unique())
missing_sources = [sid for sid in top["ID"] if sid not in have]
print(f"Missing sources to re-run: {len(missing_sources)}")
for s in missing_sources:
    print(" -", s)

# Make sure failures file has a header
if not os.path.exists("../csvs/karma_failures.csv") or os.path.getsize("../csvs/karma_failures.csv") == 0:
    with open("../csvs/karma_failures.csv", "w", newline="") as f:
        csv.writer(f).writerow(["source_id", "chunk_index", "reason", "raw_response"])

# We'll call score() per chunk. score() returns {} on no match, so for chunked work
# we want raw responses on failure - wrap in a chunk-aware helper.
def score_chunk(source_id, target_chunk):
    bio = bio_lookup.get(source_id, "")
    name = name_lookup.get(source_id, source_id)
    prompt = f"""You are analyzing the biography of {name} from A Song of Ice and Fire.
For every character ID listed, give a karma score from 1 to 10 reflecting how {name} feels about them, using the bio AND your general knowledge of the books.

1 = extremely hostile (sworn enemy, killer, betrayer)
10 = extremely friendly (closest ally, beloved family, sworn friend)

Biography:
{bio}

Score these character IDs (use them EXACTLY as JSON keys):
{', '.join(target_chunk)}

Return ONLY a JSON object like {{"ID_one": 7, "ID_two": 2}}. No prose, no code fences."""
    res = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "gemma4:e4b",
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": 0.4, "num_ctx": 16384, "num_predict": 4000},
        },
        timeout=900,
    )
    res.raise_for_status()
    text = res.json()["response"].strip()
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text)
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if not m:
        raise ValueError("no JSON object found")
    return json.loads(m.group(0)), text

write_lock = threading.Lock()

def process_missing(sid):
    row = top.loc[top["ID"] == sid].iloc[0]
    targets = [t for t in (row["outgoing_edges"] or "").split(";") if t]
    if not targets:
        return sid, 0, 0
    chunks = [targets[i:i + CHUNK] for i in range(0, len(targets), CHUNK)]
    target_set = set(targets)
    total_rows = 0
    failed_chunks = 0
    for idx, chunk in enumerate(chunks):
        try:
            scores, _ = score_chunk(sid, chunk)
        except Exception as e:
            failed_chunks += 1
            with write_lock, open("../csvs/karma_failures.csv", "a", newline="") as f:
                csv.writer(f).writerow([sid, idx, str(e), ""])
            continue
        rows = []
        for tid, s in scores.items():
            try:
                s = max(1, min(10, int(s)))
            except (TypeError, ValueError):
                continue
            if tid in target_set:
                rows.append([sid, tid, s])
        with write_lock, open("../csvs/karma_edges.csv", "a", newline="") as f:
            csv.writer(f).writerows(rows)
        total_rows += len(rows)
    return sid, total_rows, failed_chunks

with ThreadPoolExecutor(max_workers=4) as ex:
    futures = {ex.submit(process_missing, sid): sid for sid in missing_sources}
    for f in tqdm(as_completed(futures), total=len(futures)):
        sid, n, fails = f.result()
        print(f"{sid}: +{n} edges ({fails} failed chunks)")


### Reflection on iteration 2

Chunking fixed the immediate parsing problem. Instead of failing an entire source when one huge JSON response broke, the notebook could score targets in batches of 40 and record chunk-level failures separately. This made the LLM step more robust and easier to debug.

However, the improved LLM pipeline exposed a deeper data problem: some source characters had inflated target lists before the model ever saw them. The strongest example was Beron Stark. His page looked highly connected because the scrape had captured linked names from family-tree diagrams, not just narrative affiliations. The model then assigned friendly family-like scores to many distant relatives, which made Beron look artificially kind.

That means iteration 2 solved the model-output problem but not the input-data problem. The next iteration had to fix the graph construction before trusting the karma labels.

---

## Data-quality audit: family-tree inflation

After the re-run completed, a sanity-check on Beron Stark revealed a problem in the **upstream scrape**, not in the LLM.

The `out_degree` we used to pick the top-100 was built from a wiki scrape that counted **every linked character** on a page - including the boxes in ancestor and descendant **family-tree diagrams**. For dynasty-heavy pages this massively inflates the count:

- Beron Stark's prose mentions ~16 characters. His v1 `out_degree` was **88** - the extra ~70 came from his family-tree diagrams (which extend down through 7 generations all the way to Robb, Sansa, even Tyrion Lannister via Jeyne Westerling).
- The LLM was then asked to score Beron's "feelings" toward 88 mostly-Stark relatives, defaulted to "family = friendly", and produced an average karma of **7.37** - making him look like the kindest character in Westeros.

A cleaner scraper (output: `../csvs/characters_enriched_v3.csv`) excludes family-tree diagrams and keeps only the `affiliated` field. Comparing v1 vs v3 top-100:

- **88 of 100 characters overlap** - the narrative core is unchanged.
- **12 characters drop out** (their high rank was an artifact of family-tree links): Beron Stark, Loras Tyrell, Daeron II Targaryen, Daenaera Velaryon, Merrett Frey, Aemon Targaryen, Selyse Florent, Alester Florent, Valarr Targaryen, Gerold Hightower, Aemon Targaryen, Lyanna Stark.
- **12 new entrants** that were previously crowded out - all clearly narratively central: Melisandre, Varys, Doran Martell, Edmure Tully, Edmyn Tully, Ambrose Butterwell, Randyll Tarly, Rhaella Targaryen, Rhaena Targaryen, Alyn Stokeworth, Jon Hightower, Willis Fell.

### Plan to fix without re-running the LLM
For the 88 overlapping sources, we don't need to re-score anything - we just need to **drop edges that don't exist in the v3 affiliations** (i.e., the family-tree edges). Each remaining score was a per-pair judgement, so removing rows is mathematically equivalent to having scored only the cleaner edges in the first place.

For the 12 new entrants we score them fresh - a small ~12% incremental run.


### Step A - Prune family-tree edges using the v3 scrape

Load `../csvs/characters_enriched_v3.csv`, build the set of legitimate `(source, target)` pairs from its `affiliated` column, and filter `../csvs/karma_edges.csv` down to only those pairs. Output goes to `../csvs/karma_edges_v2.csv` so the original v1 file stays intact and the prune is reversible.


In [6]:
# --- Build the cleaned top-100 and a set of legitimate (source, target) pairs ---
v2 = pd.read_csv("../csvs/characters_enriched_v3.csv")
v2["affiliated_list"] = v2["affiliated"].fillna("").apply(lambda s: [t for t in s.split(";") if t])
v2["out_degree_v2"] = v2["affiliated_list"].apply(len)

# Cleaned top-100 (kept in memory - Step C re-reads the canonical ../csvs/top100_outgoing_v3.csv)
top100_v2 = (
    v2.sort_values("out_degree_v2", ascending=False)
      .head(100)[["ID", "name", "allegiance", "out_degree_v2", "affiliated"]]
      .rename(columns={"out_degree_v2": "out_degree", "affiliated": "outgoing_edges"})
      .reset_index(drop=True)
)
print(f"Cleaned top-100 computed - {len(top100_v2)} sources")

# Build the legitimate edge set: only for cleaned-top100 sources, and only their affiliated targets
top100_v2_ids = set(top100_v2["ID"])
legit_edges = set()
for _, row in v2.iterrows():
    if row["ID"] in top100_v2_ids:
        for tgt in row["affiliated_list"]:
            legit_edges.add((row["ID"], tgt))
print(f"Legitimate (source, target) pairs: {len(legit_edges):,}")

# --- Apply the filter ---
edges = pd.read_csv("../csvs/karma_edges.csv")
before = len(edges)
before_sources = edges["source_id"].nunique()

mask = [(s, t) in legit_edges for s, t in zip(edges["source_id"], edges["target_id"])]
filtered = edges[mask].reset_index(drop=True)
filtered.to_csv("../csvs/karma_edges_v2.csv", index=False)

after = len(filtered)
after_sources = filtered["source_id"].nunique()
print(f"\nEdges before prune: {before:,} ({before_sources} sources)")
print(f"Edges after  prune: {after:,} ({after_sources} sources)")
print(f"Dropped: {before - after:,} edges  ({(before-after)/before:.1%} of total)")
print(f"\nSaved to ../csvs/karma_edges_v2.csv")


top100_outgoing_v2.csv saved — 100 sources
Legitimate (source, target) pairs in v2: 10,516

Edges before prune: 10,735 (95 sources)
Edges after  prune: 8,934 (84 sources)
Dropped: 1,801 edges  (16.8% of total)

Saved to karma_edges_v2.csv


### Reflection on iteration 3A

This pruning step changes the question from "Can the LLM score these pairs?" to "Should these pairs have been scored at all?" That distinction is important. The model can only label the relationships we give it; if the upstream scrape includes family-tree artifacts, the LLM will faithfully assign scores to relationships that should not be in the analysis dataset.

By rebuilding the legitimate edge set from `../csvs/characters_enriched_v3.csv`, we kept the useful work from the first two runs where possible, but removed source-target pairs that were no longer valid under the cleaner scraper. The process is reversible because the original `../csvs/karma_edges.csv` is untouched and the pruned output is written separately as `../csvs/karma_edges_v2.csv`.


### Step B - Score the 12 new entrants

After Step A, `../csvs/karma_edges_v2.csv` covers the 88 sources that are in *both* the v1 and v3 top-100. The 12 new entrants from the cleaner scrape (Varys, Melisandre, Doran Martell, etc.) have no scores yet. Score them now using the v3 `affiliated` lists as targets, chunked into batches of 40 like the earlier re-run. Results are **appended** to `../csvs/karma_edges_v2.csv`; chunk failures go to `../csvs/karma_failures.csv`.


In [7]:
# Self-contained: defines CHUNK / score_chunk / write_lock locally so this cell
# does NOT depend on having re-run the earlier "Re-run missing sources" cell.

import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

CHUNK = 40  # targets per LLM call - keeps the JSON response short enough to parse reliably

def score_chunk(source_id, target_chunk):
    bio = bio_lookup.get(source_id, "")
    name = name_lookup.get(source_id, source_id)
    prompt = f"""You are analyzing the biography of {name} from A Song of Ice and Fire.
For every character ID listed, give a karma score from 1 to 10 reflecting how {name} feels about them, using the bio AND your general knowledge of the books.

1 = extremely hostile (sworn enemy, killer, betrayer)
10 = extremely friendly (closest ally, beloved family, sworn friend)

Biography:
{bio}

Score these character IDs (use them EXACTLY as JSON keys):
{', '.join(target_chunk)}

Return ONLY a JSON object like {{"ID_one": 7, "ID_two": 2}}. No prose, no code fences."""
    res = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "gemma4:e4b",
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": 0.4, "num_ctx": 16384, "num_predict": 4000},
        },
        timeout=900,
    )
    res.raise_for_status()
    text = res.json()["response"].strip()
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text)
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if not m:
        raise ValueError("no JSON object found")
    return json.loads(m.group(0)), text

write_lock = threading.Lock()

# Identify v3 top-100 sources that have no rows yet in karma_edges_v2.csv
v2_edges = pd.read_csv("../csvs/karma_edges_v2.csv")
have_v2 = set(v2_edges["source_id"].unique())

new_entrant_ids = [sid for sid in top100_v2["ID"] if sid not in have_v2]
print(f"New entrants to score: {len(new_entrant_ids)}")
for sid in new_entrant_ids:
    name = name_lookup.get(sid, sid)
    deg = int(top100_v2.loc[top100_v2["ID"] == sid, "out_degree"].iloc[0])
    print(f"  - {name}  ({deg} targets)")

# Build a per-source target list from v2's affiliated column
v2_targets = {
    row["ID"]: row["affiliated_list"]
    for _, row in v2.iterrows()
    if row["ID"] in set(new_entrant_ids)
}

def process_new_entrant(sid):
    targets = v2_targets.get(sid, [])
    if not targets:
        return sid, 0, 0
    chunks = [targets[i:i + CHUNK] for i in range(0, len(targets), CHUNK)]
    target_set = set(targets)
    total_rows = 0
    failed_chunks = 0
    for idx, chunk in enumerate(chunks):
        try:
            scores, _ = score_chunk(sid, chunk)
        except Exception as e:
            failed_chunks += 1
            with write_lock, open("../csvs/karma_failures.csv", "a", newline="") as f:
                csv.writer(f).writerow([sid, idx, str(e), ""])
            continue
        rows = []
        for tid, s in scores.items():
            try:
                s = max(1, min(10, int(s)))
            except (TypeError, ValueError):
                continue
            if tid in target_set:
                rows.append([sid, tid, s])
        with write_lock, open("../csvs/karma_edges_v2.csv", "a", newline="") as f:
            csv.writer(f).writerows(rows)
        total_rows += len(rows)
    return sid, total_rows, failed_chunks

with ThreadPoolExecutor(max_workers=4) as ex:
    futures = {ex.submit(process_new_entrant, sid): sid for sid in new_entrant_ids}
    for f in tqdm(as_completed(futures), total=len(futures)):
        sid, n, fails = f.result()
        print(f"{sid}: +{n} edges ({fails} failed chunks)")


New entrants to score: 16
  - Rogar Baratheon  (109 targets)
  - Daemon Velaryon  (107 targets)
  - Petyr Baelish  (105 targets)
  - Otto Hightower  (100 targets)
  - Melisandre  (77 targets)
  - Ambrose Butterwell  (76 targets)
  - Edmyn Tully  (76 targets)
  - Varys  (76 targets)
  - Doran Martell  (74 targets)
  - Edmure Tully  (74 targets)
  - Rhaena Targaryen  (73 targets)
  - Alyn Stokeworth  (71 targets)
  - Jon Hightower  (71 targets)
  - Randyll Tarly  (70 targets)
  - Rhaella Targaryen  (70 targets)
  - Willis Fell  (70 targets)


  6%|▋         | 1/16 [08:07<2:01:51, 487.40s/it]

Otto_Hightower: +100 edges (0 failed chunks)


 12%|█▎        | 2/16 [08:53<53:06, 227.57s/it]  

Daemon_Velaryon_(son_of_Aethan): +107 edges (0 failed chunks)


...

Randyll_Tarly: +70 edges (0 failed chunks)


 94%|█████████▍| 15/16 [30:38<01:40, 100.18s/it]

Rhaella_Targaryen: +70 edges (0 failed chunks)


100%|██████████| 16/16 [31:28<00:00, 118.02s/it]

Willis_Fell: +70 edges (0 failed chunks)


### Reflection on iteration 3B

After pruning, the top-100 list changed. Most of the narrative core stayed the same, but characters whose high rank came from family-tree artifacts dropped out, and cleaner high-degree characters entered. These new entrants had no karma scores because they were not part of the original first pass.

This step fills that hole without rerunning the whole dataset. The important design choice is that the notebook appends only the missing new-source scores to `karma_edges_v2.csv`. That keeps the expensive LLM work incremental and preserves a clear audit trail: original scores, pruned scores, then newly added v2 scores.

### Step C - Final cleanup: drop corrupted rows and fill remaining gaps

This step takes `../csvs/karma_edges_v2.csv` as input and produces the canonical `../csvs/karma_edges_v3.csv`. Two leftover issues from v2:

1. **16 byte-level corrupted rows** in `karma_edges_v2.csv` - scores like `5w`, `10w`, `w` - likely from the original `karma_edges.csv` being opened in an editor that mangled characters. Those rows are unrecoverable as written; we drop them so their `(source, target)` pairs become "missing" again.
2. **6 partial sources** (Ryam Redwyne, Joffrey Baratheon, Alyn Connington, Robert I Baratheon, Cersei Lannister, Robb Stark) that Step B skipped because they already had *some* rows. Their out_degree counts stuck at multiples of 40 (the chunk size), meaning earlier chunks failed silently.

Approach: re-derive each top-100 source's missing target IDs as `(v3 affiliated) − (currently present targets)`, then re-run the chunked LLM scorer **only on the missing pairs**. The output - clean v2 rows plus the new gap-fill scores - is written to `../csvs/karma_edges_v3.csv`. The v2 intermediate is left untouched.

Self-contained - does NOT depend on the earlier re-run / Step B cells being in the kernel.


In [ ]:
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

CHUNK = 40
write_lock = threading.Lock()

def score_chunk(source_id, target_chunk):
    bio = bio_lookup.get(source_id, "")
    name = name_lookup.get(source_id, source_id)
    prompt = f"""You are analyzing the biography of {name} from A Song of Ice and Fire.
For every character ID listed, give a karma score from 1 to 10 reflecting how {name} feels about them, using the bio AND your general knowledge of the books.

1 = extremely hostile (sworn enemy, killer, betrayer)
10 = extremely friendly (closest ally, beloved family, sworn friend)

Biography:
{bio}

Score these character IDs (use them EXACTLY as JSON keys):
{', '.join(target_chunk)}

Return ONLY a JSON object like {{"ID_one": 7, "ID_two": 2}}. No prose, no code fences."""
    res = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "gemma4:e4b",
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": 0.4, "num_ctx": 16384, "num_predict": 4000},
        },
        timeout=900,
    )
    res.raise_for_status()
    text = res.json()["response"].strip()
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text)
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if not m:
        raise ValueError("no JSON object found")
    return json.loads(m.group(0))

# --- 1) Drop malformed rows ---
edges = pd.read_csv("../csvs/karma_edges_v2.csv", dtype=str)
mask_ok = edges["karma_score"].str.match(r"^\d+$", na=False) & edges["karma_score"].fillna("").apply(
    lambda s: s.isdigit() and 1 <= int(s) <= 10
)
dropped_n = (~mask_ok).sum()
edges = edges[mask_ok].copy()
edges["karma_score"] = edges["karma_score"].astype(int)
edges.to_csv("../csvs/karma_edges_v3.csv", index=False)
print(f"Dropped {dropped_n} corrupted rows")
print(f"After cleanup: {len(edges):,} edges, {edges['source_id'].nunique()} sources")

# --- 2) Compute per-source missing targets ---
v2 = pd.read_csv("../csvs/characters_enriched_v3.csv")
top100_v2 = pd.read_csv("../csvs/top100_outgoing_v3.csv")
top100_ids = set(top100_v2["ID"])

existing_pairs = set(zip(edges["source_id"], edges["target_id"]))
gaps = {}
for _, row in v2.iterrows():
    sid = row["ID"]
    if sid not in top100_ids:
        continue
    affs = row["affiliated"] if pd.notna(row["affiliated"]) else ""
    targets = [t for t in affs.split(";") if t]
    missing = [t for t in targets if (sid, t) not in existing_pairs]
    if missing:
        gaps[sid] = missing

total_missing = sum(len(v) for v in gaps.values())
print(f"\nSources with gaps: {len(gaps)}")
print(f"Total (source, target) pairs to re-score: {total_missing}")
for sid, miss in sorted(gaps.items(), key=lambda x: -len(x[1]))[:10]:
    print(f"  - {sid}: {len(miss)} missing")

# --- 3) Score only the missing pairs, chunked ---
def fill_gaps(sid):
    targets = gaps.get(sid, [])
    if not targets:
        return sid, 0, 0
    chunks = [targets[i:i + CHUNK] for i in range(0, len(targets), CHUNK)]
    target_set = set(targets)
    total_rows = 0
    failed_chunks = 0
    for idx, chunk in enumerate(chunks):
        try:
            scores = score_chunk(sid, chunk)
        except Exception as e:
            failed_chunks += 1
            with write_lock, open("../csvs/karma_failures.csv", "a", newline="") as f:
                csv.writer(f).writerow([sid, idx, str(e), ""])
            continue
        rows = []
        for tid, s in scores.items():
            try:
                s = max(1, min(10, int(s)))
            except (TypeError, ValueError):
                continue
            if tid in target_set:
                rows.append([sid, tid, s])
        with write_lock, open("../csvs/karma_edges_v3.csv", "a", newline="") as f:
            csv.writer(f).writerows(rows)
        total_rows += len(rows)
    return sid, total_rows, failed_chunks

if gaps:
    with ThreadPoolExecutor(max_workers=4) as ex:
        futures = {ex.submit(fill_gaps, sid): sid for sid in gaps}
        for f in tqdm(as_completed(futures), total=len(futures)):
            sid, n, fails = f.result()
            print(f"{sid}: +{n} edges ({fails} failed chunks)")
else:
    print("\nNo gaps to fill.")


### Reflection on iteration 3C

The final cleanup handles two remaining quality issues. First, a small number of rows had corrupted score values such as `5w`, `10w`, or `w`; those cannot be interpreted as valid 1-10 scores, so the safest choice is to drop them and treat their pairs as missing. Second, some sources had partial coverage because earlier chunked calls succeeded for only part of their target list.

The final approach is the most reliable version of the pipeline: derive the expected source-target pairs from the cleaned affiliation data, compare them with the rows already scored, and ask the LLM only for the missing pairs. The result is `karma_edges_v3.csv` - the only karma edge file kept on disk, and the one all downstream notebooks read. The v1 and v2 intermediates were valuable as construction stages but are not preserved.


---

## What happens next

This notebook stops at the production of `../csvs/karma_edges_v3.csv`. All plots and analysis of the karma data - score distribution, ego networks, character-level views - live in `karma_viz.ipynb`, which reads this file as input. Keeping construction and analysis in separate notebooks means the expensive LLM step is never re-run when we only want to redraw a plot.


### Limitations

**The karma scores are not objective.** Each label is one model's interpretation of how a character feels about another, produced by a generative LLM at a non-zero temperature. A different prompt, a different model, or even a re-run of the same prompt can yield different scores. We treat the labels as a reproducible annotation step, not as a measurement.

Two further caveats this notebook does not fix. **Canon leakage**: the prompt invites the model to use its general knowledge of the books, so a score can reflect events not present in the scraped bio - labels are not strictly traceable to a quoted source. **Temporal smear**: the wiki collapses ~5,000 years into one static graph, so some scored pairs (e.g. Daenerys to Aegon I) are not meaningful relationships. Both are project-wide caveats inherited from upstream data, flagged here so that `karma_viz.ipynb` and the website can frame the scores honestly.
